# Olympians

This notebook builds a focused question-answering workflow for Greek mythology. It uses a small Persian reference dataset, a constrained system prompt, and few-shot examples to produce concise Persian answers about Olympian gods.

## Configure the Cohere API key

Run the following cell and enter your Cohere API key when prompted. The key is kept only in the current notebook process through the `CO_API_KEY` environment variable.

In [13]:
import getpass
import os

os.environ["CO_API_KEY"] = getpass.getpass()

## Greek-god reference data

The `gods.json` file contains Persian-language facts about Greek gods, including their names, descriptions, appearances, and defining features. Load this reference data and pass it to the model as context so answers remain grounded in the supplied material.

In [14]:
import json

# Read the file "gods.json" and load it into the variable "gods"
with open("gods.json", "r") as file:
    gods = json.load(file)

In [15]:
print('Total number of items:', len(gods))

Total number of items: 14


In [16]:
# Let's print the first item in the list to see what it looks like
gods[0]

{'name': 'زئوس',
 'description': 'زئوس بر کوه الیمپوس حکومت می\u200cکند و پادشاه خدایان است. او پسر کرونوس و ریا، شوهر هرا است، و 15 کودک دارد که همه آنها المپیایی هستند.',
 'appearance': 'او به عنوان یک مرد قد بلند و ستبر با لباس فاخر نشان داده شده است. او دارای موی بلند و ریش بلند است. به طور کلی، او به صورت ایستاده به تصویر درآمده است.',
 'features': 'رعد، اسلحه او و ویژگی اصلی وی است که به طور کلی در دست راست او قرار دارد. گاهی اوقات او یک عصای سلطنتی را در دست چپ خود حمل می\u200cکند. گاو و عقاب حیوانات مقدس او هستند.'}

## Create the system message

Create a system message that includes the reference data and instructs the model to answer in Persian. Responses must be no longer than four or five words. When the answer is unknown or cannot fit that limit, use the designated Persian fallback phrase exactly.

In [17]:
gods_text = json.dumps(gods, ensure_ascii=False, indent=2)
gods_text = gods_text.replace("{", "{{").replace("}", "}}")
system_message = f"""
You are an assistant that answers questions about Greek gods.

Information about the gods:
{gods_text}

Rules:
- Always answer in Persian.
- Answer in at most 4 or 5 words.
- If you don't know the answer, respond exactly with "نمی‌دانم".
- Do not provide any additional explanation.
"""

## Add few-shot examples

Few-shot examples help the model follow the required question-and-answer format. The examples below are intentionally separate from the evaluation questions and do not include the fallback answer.

In [18]:
examples = [
    {"question": "نام مادر زئوس چه بود؟", "answer": "رئا"},

    {"question": "آتنا از کدام شهر یونان محافظت می‌کرد؟", "answer": "آتن"},

    {"question": "پرسئوس کدام هیولا را کشت؟", "answer": "مدوسا"},

    {"question": "چه کسی محکوم بود آسمان را بر دوش نگه دارد؟", "answer": "اطلس"},

    {"question": "نام مادر آشیل چه بود؟", "answer": "تتیس"},

    {"question": "کدام قهرمان دوازده مأموریت دشوار انجام داد؟", "answer": "هراکلس"},

    {"question": "چه کسی آتش را از خدایان دزدید و به انسان‌ها داد؟", "answer": "پرومتئوس"},

    {"question": "چه کسی با بال‌های مومی به خورشید نزدیک شد؟", "answer": "ایکاروس"},

    {"question": "نام همسر اورفئوس چه بود؟", "answer": "ائورودیکه"},

    {"question": "چه کسی با استفاده از نخ از هزارتوی مینوتور خارج شد؟", "answer": "تسئوس"},
]

## Build the prompt template

Combine the system message, examples, and the incoming user question in a LangChain chat prompt.

In [19]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)

# Create a prompt template for each example
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

# Create the few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Create the final prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    few_shot_prompt,
    ("human", "{input}")
])

## Create the model

Configure Cohere with zero temperature so it favors consistent, high-probability answers rather than variation or creativity.

In [20]:
from langchain_cohere import ChatCohere

model = ChatCohere(temperature=0)

## Build the chain

Create a chain that sends the formatted prompt to the model and returns the response as plain text.

In [21]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

## Evaluate the model

`data/questions.csv` contains the evaluation questions. Invoke the chain in order and store each answer in the matching position. Do not add evaluation questions to the few-shot examples; doing so would invalidate the evaluation.

Cohere rate limits apply. The loop pauses after every ten requests to reduce the risk of throttling.

### Expected submission format

Store generated responses in a DataFrame named `submission` with one `answer` column. Row *i* must contain the answer for row *i* of the input questions.

In [22]:
import pandas as pd
import time

# Load the evaluation questions
questions = pd.read_csv("data/questions.csv")

# Generate responses
answers = []

for i, question in enumerate(questions["question"]):
    answer = chain.invoke({"input": question})
    answers.append(answer)

    # Manage API rate limits
    if (i + 1) % 10 == 0:
        time.sleep(65)



In [23]:
# Create the final submission DataFrame
submission = pd.DataFrame({
    "answer": answers
})

## Scoring

A submission passes when at least 85% of answers are correct. Evaluation checks whether the expected word is present in the submitted answer. Answers longer than five words are considered incorrect.

## Generate the submission bundle

Run the following cell after saving the notebook. It writes `submission.csv`, `examples.csv`, and `model_configs.json`, then packages them with the notebook in `result.zip` for the evaluation system.

In [24]:
import zipfile
import os

if not os.path.exists(os.path.join(os.getcwd(), 'Olympians.ipynb')):
    %notebook -e Olympians.ipynb

def compress(file_names): 
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

configs = {
    "messages": len(prompt.messages),
    "temperature": model.temperature
}

with open('model_configs.json', 'w') as json_file:
    json.dump(configs, json_file, indent=4)

submission.to_csv('submission.csv', index=False)
pd.DataFrame(examples).to_csv('examples.csv', index=False)

file_names = ['Olympians.ipynb', 'submission.csv',
              'examples.csv','model_configs.json' ]
compress(file_names)

File Paths:
['Olympians.ipynb', 'submission.csv', 'examples.csv', 'model_configs.json']


The generated `result.zip` can now be uploaded to the evaluation system.